# Thực nghiệm 2.5 — Tái thiết kế các hình Golub et al. (1999)

**Phụ trách:** Thành viên 5 (TV5)

- **Fig. 2:** Phân tích lân cận theo hai hướng ALL/AML và 400 lần hoán vị nhãn.
- **Fig. 3A:** Prediction Strength của Weighted Voting trên LOOCV và tập kiểm tra.
- **Fig. 4:** Khám phá lớp bằng SOM với đủ panel A–D.
- **F04 bổ sung:** Phân tích lỗi Logistic Regression theo mẫu, loại mô và nguồn.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

# Xac dinh thu muc goc du an
p = Path.cwd()
ROOT = None
while p != p.parent:
    if (p / 'src' / 'load.py').exists():
        ROOT = p
        break
    p = p.parent
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục dự án hoặc thư mục notebooks.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.load import load_golub
from src.evaluate import evaluate, _baseline_pipeline
from src.viz import save_figure
from src.viz_redesign import (
    compute_fig4, plot_fig2_neighborhood, plot_fig3a_prediction_strength,
    plot_fig4_som_redesign, plot_fig4_error_analysis,
)

print('Đã nạp các module dự án.')


## 1. Nạp dữ liệu curated

`load_golub("curated")` đọc `X_log10.parquet`. Fig. 2 dùng nhãn và biểu hiện ở 38 mẫu train. Với Fig. 4, lọc probe theo ngưỡng biến thiên năm lần và fit SOM chỉ trên 38 mẫu train. 34 mẫu test chỉ đi qua mô hình dự đoán cụm đã fit.


In [ ]:
X_cu, samples = load_golub('curated')
print(f'Curated log10: {X_cu.shape[0]} mẫu × {X_cu.shape[1]} probe')
display(samples.groupby(['split', 'class']).size().unstack())


## 2. Fig. 2 — Phân tích lân cận

Tính SNR từ 38 mẫu train. Hai panel lần lượt đếm số probe vượt ngưỡng theo hướng ALL và AML. Phân vị 95% và 99% lấy từ 400 lần hoán vị nhãn, giữ nguyên số lượng mỗi lớp. Nguồn thu thập trùng với lớp trong train, nên phép đối chiếu chưa tách được tác động của nguồn.

## 3. Fig. 4 — Khám phá lớp bằng SOM

- **A/C:** Thành phần hai cụm và bốn cụm. SOM dùng probe có mức biến thiên trên năm lần, lọc trên train theo ghi chú 33 của bài gốc.
- **B:** Weighted Voting với 20 probe, dùng nhãn SOM hai cụm. Đối chiếu PS từng mẫu với phân bố trung vị PS từ 100 lần hoán vị nhãn cụm, giữ nguyên kích thước hai cụm.
- **D:** Đánh giá sáu cặp cụm bằng LOOCV trên phần mẫu thuộc từng cặp. Mỗi lần LOOCV chọn lại probe cho bộ dự đoán. Nhãn cụm đã được xác định trên toàn train, vì vậy LOOCV đo mức phân biệt nhãn cụm đã có. PS trên test biểu thị biên phiếu dự đoán cụm. Bộ test chưa có nhãn cụm chuẩn để tính độ chính xác.


In [ ]:
som = compute_fig4(X_cu, samples)
display(som.two_clusters)
display(som.four_clusters)
display(pd.DataFrame({
    'tập': ['LOOCV', 'test'],
    'PS trung vị của SOM': [pd.Series(som.ps_cv).median(), pd.Series(som.ps_test).median()],
    'trung vị của đối chứng': pd.DataFrame(som.null_medians).median().to_list(),
}))
display(pd.DataFrame([
    {'cặp': pair, 'số mẫu': len(scores), 'PS trung vị': scores['ps'].median(),
     'đúng': scores['pred'].eq(scores['true']).sum(), 'bất định': scores['pred'].eq('uncertain').sum()}
    for pair, scores in som.pairwise.items()
]))

for mode in ('report', 'slide'):
    fig2 = plot_fig2_neighborhood(X_cu, samples, mode=mode)
    fig4 = plot_fig4_som_redesign(som, mode=mode)
    save_figure(fig2, 'F02_neighborhood_redesign', mode)
    save_figure(fig4, 'F02_som_redesign', mode)
    if mode == 'report':
        plt.show()
    plt.close(fig2)
    plt.close(fig4)


## 4. Tái thiết kế Fig 3A — Định lượng độ tin cậy dự đoán (Prediction Strength - Golub Fig. 3A)

- Nạp dữ liệu `results/metrics/golub_ps.csv` do TV1 cung cấp từ mô hình Golub Weighted Voting.
- Phân bố giá trị PS theo lớp ALL/AML trên tập huấn luyện (LOOCV, 38 mẫu) và tập kiểm tra độc lập (34 mẫu).
- Làm rõ vùng không chắc chắn (Uncertain, $PS < 0.30$) và đối chiếu trường hợp mẫu 66 và mẫu 67.

Trên tập kiểm tra, 9/10 mẫu PB được dự đoán đúng và mẫu 67 có nhãn `uncertain`. Mẫu 66 là AML, được dự đoán thành ALL với PS 0,30004.


In [ ]:
ps_df = pd.read_csv(ROOT / 'results' / 'metrics' / 'golub_ps.csv')
print('Tong so mau PS:', len(ps_df))

# Ve va xuat ban report
fig3a_rep = plot_fig3a_prediction_strength(ps_df, mode='report')
path3a_rep = save_figure(fig3a_rep, 'F03a_prediction_strength', mode='report')
plt.show()

# Ve va xuat ban slide
fig3a_sld = plot_fig3a_prediction_strength(ps_df, mode='slide')
path3a_sld = save_figure(fig3a_sld, 'F03a_prediction_strength', mode='slide')
plt.close(fig3a_sld)
print('Da xuat F03a thanh cong vao results/figures/')


## 5. F04 — Phân tích lỗi Logistic Regression trên tập kiểm tra độc lập

Dải điểm AML 0,3–0,7 là vùng tham chiếu trực quan. Ngưỡng phân loại là 0,5.

In [ ]:
pipe = _baseline_pipeline(k=50)
res_test = evaluate(pipe, X_cu, samples['class'], scheme='original_split')
preds_test = res_test['predictions']
errors = preds_test.loc[preds_test['y_pred'].ne(preds_test['y_true'])]
print(f'{len(errors)}/{len(preds_test)} mẫu phân loại sai')
display(errors)

fig4_report = plot_fig4_error_analysis(preds_test, samples, mode='report')
save_figure(fig4_report, 'F04_error_analysis', mode='report')
plt.show()

fig4_slide = plot_fig4_error_analysis(preds_test, samples, mode='slide')
save_figure(fig4_slide, 'F04_error_analysis', mode='slide')
plt.close(fig4_slide)
print('Da xuat F04_error_analysis cho ca report va slide.')

## 6. Kết luận

- **Fig. 2:** Phân tích lân cận cho thấy số probe liên hệ với ALL/AML trên tập train vượt mức ngẫu nhiên ở một số ngưỡng.
- **Fig. 3A:** LOOCV có 36/38 mẫu chắc chắn và đúng 36/36. Tập kiểm tra có 33/34 mẫu chắc chắn và đúng 32/33. Trong 10 mẫu PB, 9 mẫu đúng và mẫu 67 bất định.
- **Fig. 4:** Báo cáo thành phần cụm, PS của mô hình dự đoán cụm so với hoán vị, và PS của cả sáu cặp. Kết quả thuộc cấu hình train và seed đã nêu trên hình. Kết quả khám phá trên 72 mẫu ở mục 1.5 được báo cáo riêng.
- **F04 bổ sung:** Logistic Regression phân loại đúng 33/34 mẫu, gồm cả 10 mẫu PB. Mẫu 66 là ca phân loại sai.
